In [1]:
!pip install -q ultralytics opencv-python pyyaml tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 436.5 kB/s eta 0:00:00


In [2]:
import yaml

DATASET      = "/kaggle/input/datasets/nishkaav/urban-accessibility-dataset/Object detection.v1i.yolov8"
PATCHED_YAML = "/kaggle/working/data_patched.yaml"

cfg = {
    "train": f"{DATASET}/train/images",
    "val":   f"{DATASET}/valid/images",
    "test":  f"{DATASET}/test/images",
    "nc":    3,
    "names": ["Broken_pavement", "Missing_footpath", "Obstacle"]
}

with open(PATCHED_YAML, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print("✓ Patched yaml saved")
for k, v in cfg.items():
    print(f"  {k}: {v}")

✓ Patched yaml saved
  train: /kaggle/input/datasets/nishkaav/urban-accessibility-dataset/Object detection.v1i.yolov8/train/images
  val: /kaggle/input/datasets/nishkaav/urban-accessibility-dataset/Object detection.v1i.yolov8/valid/images
  test: /kaggle/input/datasets/nishkaav/urban-accessibility-dataset/Object detection.v1i.yolov8/test/images
  nc: 3
  names: ['Broken_pavement', 'Missing_footpath', 'Obstacle']


In [3]:
# =============================================================================
#  SCRIPT 2 — TRAIN THE YOLOv8 MODEL
#  Run AFTER Script 1 completes successfully
#  Urban Accessibility Mapping — YOLOv8 Pipeline
#
#  What happens inside each epoch (from your notes):
#    - Load / resize images to 640×640
#    - Augmentation (mosaic, flip, HSV shifts) → avoids overfitting
#    - Forward pass → feature maps at 80×80, 40×40, 20×20
#    - Compute loss: DFL box loss + BCE class loss
#    - Backprop → adjust model weights → repeat
#    - After every epoch → run validation → log mAP, precision, recall
#    - Overfitting signal: train loss ↓ but val loss ↑ → early stop
# =============================================================================

from ultralytics import YOLO
from pathlib import Path

# ── PATHS ─────────────────────────────────────────────────────────────────
PATCHED_YAML = "/kaggle/working/data_patched.yaml"
DRIVE_OUTPUT = "/kaggle/working"
# ── TRAINING CONFIG ────────────────────────────────────────────────────────
# Matched to your handwritten notes and flowchart
TRAIN_CONFIG = {
    "data"          : PATCHED_YAML,
    "model"         : "yolov8s.pt",    # small backbone — good for street-view
                                        # use yolov8n.pt if Colab runs out of RAM
    "epochs"        : 100,
    "imgsz"         : 640,
    "batch"         : 16,              # increase to 32 if GPU allows (T4 → 16 is safe)
    "freeze"        : 10,              # freeze first 10 backbone layers (transfer learning)

    # Learning rate — cosine annealing (your notes)
    "lr0"           : 0.01,            # initial LR
    "lrf"           : 0.01,            # final LR fraction → ends at lr0 × lrf = 0.0001
    "warmup_epochs" : 3,               # linear warmup for first 3 epochs

    # Early stopping (your notes: overfitting signal)
    "patience"      : 50,              # stop if val mAP doesn't improve for 50 epochs

    # Loss weights
    "box"           : 7.5,             # bounding box regression loss
    "cls"           : 0.5,             # classification loss
    "dfl"           : 1.5,             # distribution focal loss

    # Augmentation — critical for small street-view datasets
    "mosaic"        : 1.0,             # mosaic (combines 4 images) — great for small datasets
    "fliplr"        : 0.5,             # horizontal flip (streets can be either side)
    "flipud"        : 0.0,             # NO vertical flip (streets are upright)
    "degrees"       : 5.0,             # slight rotation (camera tilt)
    "translate"     : 0.1,
    "scale"         : 0.5,
    "hsv_h"         : 0.015,           # colour shift (lighting variation)
    "hsv_s"         : 0.7,
    "hsv_v"         : 0.4,

    # Output — save directly to Drive so weights survive Colab disconnect
    "project"       : DRIVE_OUTPUT,
    "name"          : "ua_model",
    "save_period"   : 10,              # checkpoint every 10 epochs
    "val"           : True,            # validate after every epoch
    "plots"         : True,            # save confusion matrix, PR curve, F1 curve
    "exist_ok"      : True,
    "verbose"       : True,
    "device"        : 0,               # GPU (Colab T4); use "cpu" if no GPU
}

print("="*60)
print("  STEP 4 — TRAINING YOLOv8")
print("="*60)
print(f"\n  Model        : {TRAIN_CONFIG['model']}")
print(f"  Epochs       : {TRAIN_CONFIG['epochs']}")
print(f"  Image size   : {TRAIN_CONFIG['imgsz']}×{TRAIN_CONFIG['imgsz']}")
print(f"  Batch size   : {TRAIN_CONFIG['batch']}")
print(f"  Frozen layers: first {TRAIN_CONFIG['freeze']} backbone layers")
print(f"  LR           : {TRAIN_CONFIG['lr0']} → {TRAIN_CONFIG['lr0']*TRAIN_CONFIG['lrf']:.4f}")
print(f"  Early stop   : patience={TRAIN_CONFIG['patience']} epochs")
print(f"  Classes      : 3 (Broken_pavement, Missing_footpath, Obstacle)")
print(f"  Weights saved: {DRIVE_OUTPUT}/ua_model/weights/")
print(f"\n  Validation happens automatically after EVERY epoch.")
print(f"  Watch for: train/box_loss ↓ while val/box_loss ↑ = overfitting\n")

# ── LOAD BASE MODEL + TRAIN ────────────────────────────────────────────────
model   = YOLO(TRAIN_CONFIG.pop("model"))   # downloads yolov8s.pt automatically
results = model.train(**TRAIN_CONFIG)

# ── PRINT FINAL RESULTS ────────────────────────────────────────────────────
BEST_PT = Path(DRIVE_OUTPUT) / "ua_model" / "weights" / "best.pt"
LAST_PT = Path(DRIVE_OUTPUT) / "ua_model" / "weights" / "last.pt"

print("\n" + "="*60)
print("  TRAINING COMPLETE")
print("="*60)
print(f"\n  best.pt → {BEST_PT}")
print(f"  last.pt → {LAST_PT}")
print(f"\n  Artifacts saved to: {DRIVE_OUTPUT}/ua_model/")
print("  Check these files:")
print("    results.png          → train/val loss curves per epoch")
print("    confusion_matrix.png → per-class prediction accuracy")
print("    PR_curve.png         → Precision-Recall curve")
print("    F1_curve.png         → F1 vs confidence threshold")
print(f"\n  → Now run Script 3 (Validate) then Script 4 (Test)")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
  STEP 4 — TRAINING YOLOv8

  Model        : yolov8s.pt
  Epochs       : 100
  Image size   : 640×640
  Batch size   : 16
  Frozen layers: first 10 backbone layers
  LR           : 0.01 → 0.0001
  Early stop   : patience=50 epochs
  Classes      : 3 (Broken_pavement, Missing_footpath, Obstacle)
  Weights saved: /kaggle/working/ua_model/weights/

  Validation happens automatically after EVERY epoch.
  Watch for: train/box_loss ↓ while val/box_loss ↑ = overfitting

Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False,

In [4]:
!pip uninstall torch torchvision torchaudio ultralytics -y -q
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install ultralytics -q
print("Done!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 70.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 46.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 110.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 33.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 15.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 9.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 MB 13.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 905.2/905.2 MB 774.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
import torch
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
from ultralytics import YOLO
print("Ultralytics: OK")

Torch: 2.10.0+cu128
CUDA: True
Ultralytics: OK


In [6]:
# =============================================================================
#  SCRIPT 2 — TRAIN THE YOLOv8 MODEL
#  Run AFTER Script 1 completes successfully
#  Urban Accessibility Mapping — YOLOv8 Pipeline
#
#  What happens inside each epoch (from your notes):
#    - Load / resize images to 640×640
#    - Augmentation (mosaic, flip, HSV shifts) → avoids overfitting
#    - Forward pass → feature maps at 80×80, 40×40, 20×20
#    - Compute loss: DFL box loss + BCE class loss
#    - Backprop → adjust model weights → repeat
#    - After every epoch → run validation → log mAP, precision, recall
#    - Overfitting signal: train loss ↓ but val loss ↑ → early stop
# =============================================================================

from ultralytics import YOLO
from pathlib import Path

# ── PATHS ─────────────────────────────────────────────────────────────────
# Script 2 — change these two lines:
PATCHED_YAML = "/kaggle/working/data_patched.yaml"
DRIVE_OUTPUT = "/kaggle/working"
# ── TRAINING CONFIG ────────────────────────────────────────────────────────
# Matched to your handwritten notes and flowchart
TRAIN_CONFIG = {
    "data"          : PATCHED_YAML,
    "model"         : "yolov8s.pt",    # small backbone — good for street-view
                                        # use yolov8n.pt if Colab runs out of RAM
    "epochs"        : 100,
    "imgsz"         : 640,
    "batch"         : 16,              # increase to 32 if GPU allows (T4 → 16 is safe)
    "freeze"        : 10,              # freeze first 10 backbone layers (transfer learning)

    # Learning rate — cosine annealing (your notes)
    "lr0"           : 0.01,            # initial LR
    "lrf"           : 0.01,            # final LR fraction → ends at lr0 × lrf = 0.0001
    "warmup_epochs" : 3,               # linear warmup for first 3 epochs

    # Early stopping (your notes: overfitting signal)
    "patience"      : 50,              # stop if val mAP doesn't improve for 50 epochs

    # Loss weights
    "box"           : 7.5,             # bounding box regression loss
    "cls"           : 1.5,             # classification loss
    "dfl"           : 1.5,             # distribution focal loss

    # Augmentation — critical for small street-view datasets
    "mosaic"        : 1.0,             # mosaic (combines 4 images) — great for small datasets
    "fliplr"        : 0.5,             # horizontal flip (streets can be either side)
    "flipud"        : 0.0,             # NO vertical flip (streets are upright)
    "degrees"       : 5.0,             # slight rotation (camera tilt)
    "translate"     : 0.1,
    "scale"         : 0.5,
    "hsv_h"         : 0.015,           # colour shift (lighting variation)
    "hsv_s"         : 0.7,
    "hsv_v"         : 0.4,

    # Output — save directly to Drive so weights survive Colab disconnect
    "project"       : DRIVE_OUTPUT,
    "name"          : "ua_model",
    "save_period"   : 10,              # checkpoint every 10 epochs
    "val"           : True,            # validate after every epoch
    "plots"         : True,            # save confusion matrix, PR curve, F1 curve
    "exist_ok"      : True,
    "verbose"       : True,
    "device"        : 0,               # GPU (Colab T4); use "cpu" if no GPU
}

print("="*60)
print("  STEP 4 — TRAINING YOLOv8")
print("="*60)
print(f"\n  Model        : {TRAIN_CONFIG['model']}")
print(f"  Epochs       : {TRAIN_CONFIG['epochs']}")
print(f"  Image size   : {TRAIN_CONFIG['imgsz']}×{TRAIN_CONFIG['imgsz']}")
print(f"  Batch size   : {TRAIN_CONFIG['batch']}")
print(f"  Frozen layers: first {TRAIN_CONFIG['freeze']} backbone layers")
print(f"  LR           : {TRAIN_CONFIG['lr0']} → {TRAIN_CONFIG['lr0']*TRAIN_CONFIG['lrf']:.4f}")
print(f"  Early stop   : patience={TRAIN_CONFIG['patience']} epochs")
print(f"  Classes      : 3 (Broken_pavement, Missing_footpath, Obstacle)")
print(f"  Weights saved: {DRIVE_OUTPUT}/ua_model/weights/")
print(f"\n  Validation happens automatically after EVERY epoch.")
print(f"  Watch for: train/box_loss ↓ while val/box_loss ↑ = overfitting\n")

# ── LOAD BASE MODEL + TRAIN ────────────────────────────────────────────────
model   = YOLO(TRAIN_CONFIG.pop("model"))   # downloads yolov8s.pt automatically
results = model.train(**TRAIN_CONFIG)

# ── PRINT FINAL RESULTS ────────────────────────────────────────────────────
BEST_PT = Path(DRIVE_OUTPUT) / "ua_model" / "weights" / "best.pt"
LAST_PT = Path(DRIVE_OUTPUT) / "ua_model" / "weights" / "last.pt"

print("\n" + "="*60)
print("  TRAINING COMPLETE")
print("="*60)
print(f"\n  best.pt → {BEST_PT}")
print(f"  last.pt → {LAST_PT}")
print(f"\n  Artifacts saved to: {DRIVE_OUTPUT}/ua_model/")
print("  Check these files:")
print("    results.png          → train/val loss curves per epoch")
print("    confusion_matrix.png → per-class prediction accuracy")
print("    PR_curve.png         → Precision-Recall curve")
print("    F1_curve.png         → F1 vs confidence threshold")
print(f"\n  → Now run Script 3 (Validate) then Script 4 (Test)")


  STEP 4 — TRAINING YOLOv8

  Model        : yolov8s.pt
  Epochs       : 100
  Image size   : 640×640
  Batch size   : 16
  Frozen layers: first 10 backbone layers
  LR           : 0.01 → 0.0001
  Early stop   : patience=50 epochs
  Classes      : 3 (Broken_pavement, Missing_footpath, Obstacle)
  Weights saved: /kaggle/working/ua_model/weights/

  Validation happens automatically after EVERY epoch.
  Watch for: train/box_loss ↓ while val/box_loss ↑ = overfitting

Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_patched.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, e

In [7]:
# =============================================================================
#  SCRIPT 3 — VALIDATE + TEST
#  Run AFTER Script 2 (training) completes
#  Urban Accessibility Mapping — YOLOv8 Pipeline
#
#  Validation = measures model on the val split (same split used during training)
#  Test        = measures model on the held-out test split (unseen data)
#
#  Metrics explained (from your notes):
#    mAP50     → mean Average Precision at IoU=0.50 (main metric, aim for >0.5)
#    mAP50-95  → mAP averaged over IoU=0.50→0.95 (stricter, harder to get high)
#    Precision → of all boxes predicted, how many were correct?
#    Recall    → of all real boxes, how many did the model find?
# =============================================================================

import json
from pathlib import Path
from ultralytics import YOLO

# ── PATHS ─────────────────────────────────────────────────────────────────
PATCHED_YAML = "/kaggle/working/data_patched.yaml"
BEST_PT      = "/kaggle/working/ua_model/weights/best.pt"
DRIVE_OUTPUT = "/kaggle/working"

CLASS_NAMES = ["Broken_pavement", "Missing_footpath", "Obstacle"]

model = YOLO(BEST_PT)

# =============================================================================
#  PART A — VALIDATION (val split)
#  This is the split the model saw during training — sanity check
# =============================================================================
print("="*60)
print("  STEP 5A — VALIDATION (val split)")
print("="*60)
print("\n  Running on: valid/images")
print("  Purpose: confirm the model learned correctly\n")

val_metrics = model.val(
    data    = PATCHED_YAML,
    split   = "val",
    imgsz   = 640,
    batch   = 16,
    plots   = True,
    verbose = True,
    project = DRIVE_OUTPUT,
    name    = "ua_model_val_results",
    exist_ok= True,
)

print(f"\n  ── VALIDATION RESULTS ──")
print(f"  mAP50       : {val_metrics.box.map50:.4f}   (aim for >0.50)")
print(f"  mAP50-95    : {val_metrics.box.map:.4f}   (stricter metric)")
print(f"  Precision   : {val_metrics.box.mp:.4f}   (how many predictions were right)")
print(f"  Recall      : {val_metrics.box.mr:.4f}   (how many real objects found)")

# Per-class breakdown
print(f"\n  Per-class mAP50:")
if hasattr(val_metrics.box, 'ap_class_index') and val_metrics.box.ap is not None:
    for i, ap in enumerate(val_metrics.box.ap50):
        name = CLASS_NAMES[i] if i < len(CLASS_NAMES) else str(i)
        bar  = "█" * int(ap * 20)
        print(f"    {name:25s}: {ap:.4f}  {bar}")

val_results = {
    "split":     "val",
    "mAP50":     val_metrics.box.map50,
    "mAP50-95":  val_metrics.box.map,
    "precision": val_metrics.box.mp,
    "recall":    val_metrics.box.mr,
}


# =============================================================================
#  PART B — TEST (test split — HELD-OUT, unseen during training)
#  This is the true measure of how well the model generalises
# =============================================================================
print("\n" + "="*60)
print("  STEP 5B — TEST (test split — unseen data)")
print("="*60)
print("\n  Running on: test/images")
print("  Purpose: final unbiased evaluation on data the model NEVER saw\n")

test_metrics = model.val(
    data    = PATCHED_YAML,
    split   = "test",
    imgsz   = 640,
    batch   = 16,
    plots   = True,
    verbose = True,
    project = DRIVE_OUTPUT,
    name    = "ua_model_test_results",
    exist_ok= True,
)

print(f"\n  ── TEST RESULTS ──")
print(f"  mAP50       : {test_metrics.box.map50:.4f}   (aim for >0.50)")
print(f"  mAP50-95    : {test_metrics.box.map:.4f}   (stricter metric)")
print(f"  Precision   : {test_metrics.box.mp:.4f}   (how many predictions were right)")
print(f"  Recall      : {test_metrics.box.mr:.4f}   (how many real objects found)")

print(f"\n  Per-class mAP50:")
if hasattr(test_metrics.box, 'ap_class_index') and test_metrics.box.ap is not None:
    for i, ap in enumerate(test_metrics.box.ap50):
        name = CLASS_NAMES[i] if i < len(CLASS_NAMES) else str(i)
        bar  = "█" * int(ap * 20)
        print(f"    {name:25s}: {ap:.4f}  {bar}")

test_results = {
    "split":     "test",
    "mAP50":     test_metrics.box.map50,
    "mAP50-95":  test_metrics.box.map,
    "precision": test_metrics.box.mp,
    "recall":    test_metrics.box.mr,
}


# =============================================================================
#  COMPARISON SUMMARY
# =============================================================================
print("\n" + "="*60)
print("  VALIDATION vs TEST COMPARISON")
print("="*60)
print(f"\n  {'Metric':15s}  {'Val':>8s}  {'Test':>8s}  Note")
print(f"  {'-'*55}")
print(f"  {'mAP50':15s}  {val_results['mAP50']:>8.4f}  {test_results['mAP50']:>8.4f}"
      f"  {'✓ similar = generalises well' if abs(val_results['mAP50'] - test_results['mAP50']) < 0.10 else '⚠ big gap = overfitting'}")
print(f"  {'mAP50-95':15s}  {val_results['mAP50-95']:>8.4f}  {test_results['mAP50-95']:>8.4f}")
print(f"  {'Precision':15s}  {val_results['precision']:>8.4f}  {test_results['precision']:>8.4f}")
print(f"  {'Recall':15s}  {val_results['recall']:>8.4f}  {test_results['recall']:>8.4f}")

print(f"""
  How to read these results:
  ─────────────────────────────────────────────────
  mAP50 > 0.70        → Good model
  mAP50 0.50–0.70     → Acceptable, more data helps
  mAP50 < 0.50        → Needs more labeled data or longer training
  Val ≈ Test          → Model generalises well (good)
  Val >> Test         → Overfitting — add augmentation or more data
  Precision low       → Many false positives (lower conf threshold)
  Recall low          → Missing detections (raise conf threshold)
""")

# Save metrics to Drive
all_metrics = {"validation": val_results, "test": test_results}
metrics_path = f"{DRIVE_OUTPUT}/ua_model_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(all_metrics, f, indent=2)

print(f"  Metrics saved → {metrics_path}")
print(f"  Val plots     → {DRIVE_OUTPUT}/ua_model_val_results/")
print(f"  Test plots    → {DRIVE_OUTPUT}/ua_model_test_results/")
print(f"\n  → Now run Script 4 to detect on ALL dataset images\n")

  STEP 5A — VALIDATION (val split)

  Running on: valid/images
  Purpose: confirm the model learned correctly

Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,126,745 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 99.5±23.6 MB/s, size: 85.2 KB)
val: Scanning /kaggle/input/datasets/nishkaav/urban-accessibility-dataset/Object detection.v1i.yolov8/valid/labels... 87 images, 18 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 87/87 1.0Kit/s 0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/nishkaav/urban-accessibility-dataset/Object detection.v1i.yolov8/valid is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 2.8it/s 2.2s
                   all         87         75      0.133      0.278      0.131     0.0578
       Broken_pavement         50         50      0.133        0.4      0.1

In [8]:
# =============================================================================
#  SCRIPT 4 — DETECTION ON ALL IMAGES (train + valid + test)
#  Run AFTER Script 3 (validate + test)
#  Urban Accessibility Mapping — YOLOv8 Pipeline
#
#  Flowchart certainty routing:
#    High   (≥ 0.25) → Accept Detection → draw bbox → save as map data
#    Medium (≥ 0.15) → Send for Human Check → human review list
#    Low    (< 0.15) → Discard
#
#  Output:
#    high_confidence/   → annotated images WITH bounding boxes
#    human_review/      → original images for human annotation
#    accessibility_map_data.json  → all accepted detections (GPS-ready)
#    human_review_list.json       → medium confidence queue
#    detection_summary.json       → stats breakdown
# =============================================================================

import cv2, json, shutil
from pathlib import Path
from collections import Counter
from tqdm import tqdm
from ultralytics import YOLO

# ── PATHS ─────────────────────────────────────────────────────────────────
BEST_PT      = "/kaggle/working/ua_model/weights/best.pt"
DATASET_ROOT = Path("/kaggle/input/datasets/nishkaav/urban-accessibility-dataset/Object detection.v1i.yolov8")
OUTPUT_DIR   = Path("/kaggle/working/detections_all")

# ── CLASS NAMES (must match data.yaml order exactly) ──────────────────────
CLASS_NAMES = ["Broken_pavement", "Missing_footpath", "Obstacle"]

# ── BOUNDING BOX COLORS (BGR) ─────────────────────────────────────────────
COLORS = {
    0: (0,   165, 255),   # Broken_pavement  → orange
    1: (255, 0,   180),   # Missing_footpath → purple
    2: (0,   200, 200),   # Obstacle         → cyan
}

# ── CONFIDENCE THRESHOLDS (certainty score routing from flowchart) ─────────
CONF_HIGH   = 0.25   # Accept Detection → bbox drawn → map data
CONF_MEDIUM = 0.15   # Send for Human Check

# ── OUTPUT DIRS ───────────────────────────────────────────────────────────
HIGH_DIR = OUTPUT_DIR / "high_confidence"
MED_DIR  = OUTPUT_DIR / "human_review"
for d in [HIGH_DIR, MED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── COLLECT ALL IMAGES (train + valid + test) ──────────────────────────────
IMAGE_DIRS = [
    DATASET_ROOT / "train" / "images",
    DATASET_ROOT / "valid" / "images",
    DATASET_ROOT / "test"  / "images",
]

all_imgs = []
print("="*60)
print("  SCRIPT 4 — DETECTION ON ALL IMAGES")
print("="*60)
print(f"\n  Collecting images from all splits:")
for img_dir in IMAGE_DIRS:
    imgs = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png"))
    split_name = img_dir.parent.name
    print(f"    {split_name:6s}: {len(imgs):5d} images  ← {img_dir}")
    all_imgs.extend(imgs)

print(f"\n  Total: {len(all_imgs)} images")
print(f"  Thresholds: HIGH ≥ {CONF_HIGH}  |  MEDIUM ≥ {CONF_MEDIUM}  |  LOW < {CONF_MEDIUM}")
print(f"  Output: {OUTPUT_DIR}\n")

# ── LOAD MODEL ────────────────────────────────────────────────────────────
model = YOLO(BEST_PT)
print(f"  Model loaded: {BEST_PT}\n")

# ── RUN DETECTION ─────────────────────────────────────────────────────────
map_data    = []
review      = []
stats       = Counter()
class_stats = Counter()   # accepted detections per class
split_stats = {}           # per-split accepted image count

for img_path in tqdm(all_imgs, desc="  Detecting"):
    split = img_path.parent.parent.name   # "train", "valid", or "test"

    result = model.predict(
        source  = str(img_path),
        conf    = CONF_MEDIUM,   # get everything above minimum
        iou     = 0.45,
        verbose = False,
    )[0]

    if result.boxes is None or len(result.boxes) == 0:
        stats["no_detection"] += 1
        continue

    boxes   = result.boxes.xyxy.cpu().numpy()
    confs   = result.boxes.conf.cpu().numpy()
    cls_ids = result.boxes.cls.cpu().numpy().astype(int)
    img     = cv2.imread(str(img_path))
    fname   = img_path.name

    high_dets = []
    med_dets  = []

    for box, conf, cid in zip(boxes, confs, cls_ids):
        x1, y1, x2, y2 = map(int, box)
        name = CLASS_NAMES[cid] if cid < len(CLASS_NAMES) else str(cid)
        det  = {
            "class_id":   int(cid),
            "class_name": name,
            "confidence": round(float(conf), 4),
            "bbox_xyxy":  [x1, y1, x2, y2],
        }
        if conf >= CONF_HIGH:
            high_dets.append(det)
            stats["accepted"] += 1
            class_stats[name] += 1
            split_stats[split] = split_stats.get(split, 0) + 1
        else:
            med_dets.append(det)
            stats["human_review"] += 1

    # ── HIGH CONFIDENCE: draw bounding boxes ──────────────────────────────
    if high_dets:
        vis = img.copy()
        for det in high_dets:
            x1, y1, x2, y2 = det["bbox_xyxy"]
            color     = COLORS.get(det["class_id"], (255, 255, 0))
            label_str = f"{det['class_name']} {det['confidence']:.2f}"

            # Draw box
            cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)

            # Draw label background
            (tw, th), _ = cv2.getTextSize(
                label_str, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2
            )
            cv2.rectangle(vis, (x1, y1 - th - 10), (x1 + tw + 6, y1), color, -1)

            # Draw label text
            cv2.putText(
                vis, label_str, (x1 + 3, y1 - 4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2
            )

        cv2.imwrite(str(HIGH_DIR / fname), vis)

        map_data.append({
            "filename":   fname,
            "split":      split,
            "lat":        None,    # attach GPS here later
            "lon":        None,
            "detections": high_dets,
        })

    # ── MEDIUM CONFIDENCE: copy for human review ───────────────────────────
    if med_dets:
        shutil.copy(str(img_path), MED_DIR / fname)
        review.append({
            "filename":   fname,
            "split":      split,
            "detections": med_dets,
            "status":     "pending_review",
        })

# ── SAVE JSON OUTPUTS ─────────────────────────────────────────────────────
map_json    = OUTPUT_DIR / "accessibility_map_data.json"
review_json = OUTPUT_DIR / "human_review_list.json"

with open(map_json, "w") as f:
    json.dump(map_data, f, indent=2)
with open(review_json, "w") as f:
    json.dump(review, f, indent=2)

# ── SUMMARY REPORT ────────────────────────────────────────────────────────
summary = {
    "total_images":    len(all_imgs),
    "accepted":        stats["accepted"],
    "human_review":    stats["human_review"],
    "no_detection":    stats["no_detection"],
    "images_with_boxes": len(map_data),
    "per_class":       dict(class_stats),
    "per_split":       split_stats,
    "thresholds":      {"high": CONF_HIGH, "medium": CONF_MEDIUM},
}
with open(OUTPUT_DIR / "detection_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"\n{'='*60}")
print(f"  DETECTION COMPLETE — ALL {len(all_imgs)} IMAGES")
print(f"{'='*60}")
print(f"\n  Accepted  (≥{CONF_HIGH})  : {stats['accepted']:6d} detections")
print(f"  Human Review (≥{CONF_MEDIUM}): {stats['human_review']:6d} detections")
print(f"  No detection        : {stats['no_detection']:6d} images")
print(f"  Images with boxes   : {len(map_data):6d} images")

print(f"\n  Per-class breakdown (accepted detections):")
for cls, cnt in class_stats.most_common():
    bar = "█" * min(cnt // 20 + 1, 30)
    print(f"    {cls:25s}: {cnt:5d}  {bar}")

print(f"\n  Per-split breakdown (images with accepted detections):")
for split, cnt in split_stats.items():
    print(f"    {split:8s}: {cnt:5d} images")

print(f"""
  Output files:
    Annotated images  → {HIGH_DIR}/
    Review images     → {MED_DIR}/
    Map data (JSON)   → {map_json}
    Review list       → {review_json}
    Summary           → {OUTPUT_DIR}/detection_summary.json

  Next steps:
    1. Review human_review/ images and correct labels
    2. Add corrected labels back to training data (flowchart: Add to Training Data)
    3. Retrain with expanded dataset for better accuracy
    4. Load accessibility_map_data.json into Leaflet/OpenStreetMap
""")

  SCRIPT 4 — DETECTION ON ALL IMAGES

    train :  1208 images  ← /kaggle/input/datasets/nishkaav/urban-accessibility-dataset/Object detection.v1i.yolov8/train/images
    valid :    87 images  ← /kaggle/input/datasets/nishkaav/urban-accessibility-dataset/Object detection.v1i.yolov8/valid/images
    test  :    84 images  ← /kaggle/input/datasets/nishkaav/urban-accessibility-dataset/Object detection.v1i.yolov8/test/images

  Total: 1379 images
  Thresholds: HIGH ≥ 0.25  |  MEDIUM ≥ 0.15  |  LOW < 0.15
  Output: /kaggle/working/detections_all

  Model loaded: /kaggle/working/ua_model/weights/best.pt



  Detecting: 100%|██████████| 1379/1379 [00:26<00:00, 51.92it/s]


  DETECTION COMPLETE — ALL 1379 IMAGES

  Accepted  (≥0.25)  :   1836 detections
  Human Review (≥0.15):    490 detections
  No detection        :    133 images
  Images with boxes   :   1200 images

  Per-class breakdown (accepted detections):
    Broken_pavement          :  1375  ██████████████████████████████
    Missing_footpath         :   308  ████████████████
    Obstacle                 :   153  ████████

  Per-split breakdown (images with accepted detections):
    train   :  1623 images
    valid   :   109 images
    test    :   104 images

  Output files:
    Annotated images  → /kaggle/working/detections_all/high_confidence/
    Review images     → /kaggle/working/detections_all/human_review/
    Map data (JSON)   → /kaggle/working/detections_all/accessibility_map_data.json
    Review list       → /kaggle/working/detections_all/human_review_list.json
    Summary           → /kaggle/working/detections_all/detection_summary.json

  Next steps:
    1. Review human_review/ imag

In [9]:
import shutil

# Zip the model weights
shutil.make_archive("/kaggle/working/ua_model_weights", "zip", 
                    "/kaggle/working/ua_model/weights")

# Zip all detection results
shutil.make_archive("/kaggle/working/detections_all_output", "zip",
                    "/kaggle/working", "detections_all")

print("Done! Now click Save Version (top right) to persist outputs.")

Done! Now click Save Version (top right) to persist outputs.
